In [1]:
import cftime
import pandas as pd
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.ticker as mticker
from scipy.ndimage import label
from pathlib import Path
import pyarrow as pa
from collections import Counter
import regionmask
import matplotlib.patheffects as path_effects 
import cartopy.io.shapereader as shpreader
import seaborn as sns
from scipy import stats

/home/michsh/Jupyter_Env/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Ticks and Plot Features

In [2]:
import numpy as np

def generate_custom_ticks_1(grid_values):
    # 1. Get exact data limits
    exact_min = np.nanmin(grid_values)
    exact_max = np.nanmax(grid_values)
    
    if np.isnan(exact_min): 
        exact_min, exact_max = -7.0, 7.0
        
    # 2. Find the largest absolute value to force symmetry around 0
    max_abs = int(np.ceil(max(abs(exact_min), abs(exact_max))))
    
    # Force a minimum symmetric boundary (e.g., if max change is only 1, force a clean map)
    if max_abs == 0: 
        max_abs = 1
        
    # 3. Establish symmetric exact visual boundaries for pcolormesh (vmin / vmax)
    exact_min_sym = -max_abs
    exact_max_sym = max_abs
    
    # 4. Generate whole-number ticks from -max_abs to +max_abs
    # We step by 1 or 2 depending on how wide the span is to keep the colorbar clean
    step = 1
    custom_ticks = list(range(-max_abs, max_abs + 1, step))
    
    # 5. Create clean string labels without decimal points
    tick_labels = [f"{t}" for t in custom_ticks]
    
    return custom_ticks, tick_labels, exact_max_sym, exact_min_sym

In [3]:
import numpy as np

def generate_custom_ticks_05(grid_values):
    # 1. Get exact data limits
    exact_min = np.nanmin(grid_values)
    exact_max = np.nanmax(grid_values)
    
    if np.isnan(exact_min): exact_min, exact_max = 0.0, 2.0
    
    # 2. Round outward to the nearest 0.5 step to establish the uniform grid
    grid_start = np.floor(exact_min * 2) / 2
    grid_end = np.ceil(exact_max * 2) / 2
    
    # 3. Generate the full uniform step array (inclusive of grid_end)
    full_grid = np.arange(grid_start, grid_end + 0.1, 0.5)
    
    # 4. Slice off the fake outer boundaries to leave only the true middle ticks
    middle_ticks = list(full_grid[1:-1])
    
    # 5. Group together: [Exact Min, Middle Ticks..., Exact Max]
    custom_ticks = middle_ticks
    
    # 6. Generate text labels matching the spacing rules
    tick_labels = []
    for t in middle_ticks:
        if t % 1 == 0:
            tick_labels.append(f"{t:.1f}") # Pure whole numbers get no decimals
        else:
            tick_labels.append(f"{t:.1f}")  # Half-steps get 1 decimal place (e.g., 2.5)
            
    return custom_ticks, tick_labels, exact_max, exact_min

In [4]:
import numpy as np

def generate_custom_ticks_02(grid_values):
    # 1. Get exact data limits
    exact_max = np.nanmax(grid_values)
    
    # 2. Round outward to the nearest 0.5 step to establish the uniform grid
    grid_start = np.floor(-exact_max * 2) / 2
    grid_end = np.ceil(exact_max * 2) / 2
    
    # 3. Generate the full uniform step array (inclusive of grid_end)
    full_grid = np.arange(grid_start, grid_end + 0.1, .1)
    
    # 4. Slice off the fake outer boundaries to leave only the true middle ticks
    middle_ticks = list(full_grid[1:-1])
    
    # 5. Group together: [Exact Min, Middle Ticks..., Exact Max]
    custom_ticks = [-exact_max] + middle_ticks + [exact_max]
    
    # 6. Generate text labels matching the spacing rules
    tick_labels = []
    for t in custom_ticks:
        if t % 1 == 0:
            tick_labels.append(f"{t:.1f}") # Pure whole numbers get no decimals
        else:
            tick_labels.append(f"{t:.1f}")  # Half-steps get 1 decimal place (e.g., 2.5)
            
    return custom_ticks, tick_labels, exact_max, -exact_max

In [5]:

# --- STEP 1: Load Canada and Mexico Geometries once ---
shpfilename = shpreader.natural_earth(resolution='50m', category='cultural', name='admin_0_countries')
reader = shpreader.Reader(shpfilename)
records = reader.records()

canada_geom = None
mexico_geom = None

for record in records:
    country_name = record.attributes.get('NAME')
    if country_name == 'Canada':
        canada_geom = record.geometry
    elif country_name == 'Mexico':
        mexico_geom = record.geometry

# Load 50m Lakes Shapefile from Natural Earth
lakes_shp = shpreader.natural_earth(resolution='50m', category='physical', name='lakes')
lakes_reader = shpreader.Reader(lakes_shp)

# Names of the Great Lakes to isolate
great_lakes_names = {'Lake Superior', 'Lake Michigan', 'Lake Huron', 'Lake Erie', 'Lake Ontario'}

# Filter out only the Great Lakes geometries
great_lakes_geoms = []
for record in lakes_reader.records():
    # 'name' is the attribute key for the lake's name in Natural Earth
    lake_name = record.attributes.get('name')
    if lake_name in great_lakes_names:
        great_lakes_geoms.append(record.geometry)

# Use Natural Earth's defined regions for US States (50m resolution)
us_states = regionmask.defined_regions.natural_earth_v5_0_0.us_states_50

# HIST/FTR Thirstwave File

In [6]:
# Define file paths
hist = '/data1/michsh/CSV/HIST_derived_metrics_2.csv'
ftr = '/data1/michsh/CSV/FUT_derived_metrics_2.csv'

# Thirstwave Characteristics HIST/FTR AVG

## Duration

In [ ]:
import gc
import numpy as np
import pandas as pd

# 1. Define explicit lightweight dtypes (Cuts RAM usage in half)
dtypes = {
    'AMOC': 'int8',
    'member': 'int16',
    'lat': 'float32',
    'lon': 'float32',
    'duration': 'float32',
}

cols_to_keep = list(dtypes.keys())

# ==========================================
# STEP 1: Process HIST Baseline
# ==========================================
hist_chunks = []
for chunk in pd.read_csv(hist, chunksize=200000, usecols=cols_to_keep, dtype=dtypes):
    chunk = chunk[chunk['duration'] > 0]
    hist_chunks.append(chunk)

# Merge HIST chunks
hist_dur_events = (
    pd.concat(hist_chunks)
    .groupby(['AMOC', 'member', 'lat', 'lon'])['duration']
    .reset_index()
)

# # FREE MEMORY: Drop chunk list immediately
# del hist_chunks
# gc.collect()

# # Aggregate HIST to spatial grid means (shrinks millions of rows into a small map)
# hist_member_means = (
#     hist_df_events
#     .groupby(['lat', 'lon', 'AMOC', 'member'])['duration']
#     .mean()
#     .reset_index()
# )

# # ==========================================
# # STEP 2: Process FUT Scenario
# # ==========================================
# ftr_chunks = []
# for chunk in pd.read_csv(
#     '/data1/michsh/CSV/FUT_derived_metrics_2.csv',
#     chunksize=200000,
#     usecols=cols_to_keep,
#     dtype=dtypes,
# ):
#     chunk = chunk[chunk['duration'] > 0]
#     ftr_chunks.append(chunk)

# # Merge FUT chunks
# ftr_dur_events = pd.concat(ftr_chunks, ignore_index=True)

# # FREE MEMORY: Drop chunk list
# del ftr_chunks
# gc.collect()

# # Aggregate FUT to spatial grid means
# ftr_member_means = (
#     ftr_df_events
#     .groupby(['lat', 'lon', 'AMOC', 'member'])['duration']
#     .mean()
#     .reset_index()
# )

AttributeError: 'SeriesGroupBy' object has no attribute 'reset_index'

In [ ]:
hist_dur_events

In [ ]:
# --- 3. Calculate the Mean Across the 80 Ensemble Members ---
# Now we average over the 'AMOC' and 'member' dimensions to get one value per grid box.
hist_ensemble_mean_duration = (
    hist_member_means
    .groupby(['lat', 'lon'])['duration']
    .mean()
    .reset_index(name='mean_dur')
)

In [ ]:
hist_ensemble_mean_duration

,lat,lon,mean_dur
0,24.031414,235.00,4.194825
1,24.031414,236.25,4.125054
2,24.031414,237.50,4.079734
3,24.031414,238.75,4.052772
4,24.031414,240.00,4.032527
...,...,...,...
1405,51.361256,287.50,3.744724
1406,51.361256,288.75,3.712066
1407,51.361256,290.00,3.700264
1408,51.361256,291.25,3.653153


In [ ]:
del hist_dur_events
gc.collect()

del ftr_dur_events
gc.collect()

NameError: name 'hist_dur_events' is not defined